## 1. Confirm Kaggle Input Path

Before loading any data, this cell walks `/kaggle/input` to print the exact mounted file 
paths. Kaggle dataset mount paths can vary slightly depending on how a dataset was added, 
so this step avoids hardcoding a path that might not match what's actually available in 
this session.

In [1]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/datasets/eswarchandt/phishing-website-detector/phishing.txt
/kaggle/input/datasets/eswarchandt/phishing-website-detector/phishing.csv


## 2. Load Data and Subset to Selected Features

This notebook trains on the **top-7 features** identified in the prior feature importance 
extraction stage ([see notebook](https://www.kaggle.com/code/azimehobadiah/url-phising-dataset-feature-importance/)), 
rather than the full 30-feature set. The selected features — `HTTPS`, `AnchorURL`, 
`PrefixSuffix-`, `ServerFormHandler`, `WebsiteTraffic`, `GoogleIndex`, `DNSRecording` — were 
ranked using XGBoost gain-based importance and cross-validated against SHAP values.

The target column `class` is remapped from its original `{-1, 1}` encoding to `{0, 1}`, 
where `1 = legitimate` and `0 = phishing`, for compatibility with scikit-learn/XGBoost 
binary classifiers.

**Note on positive class convention:** since `0 = phishing` in this encoding, all scoring 
metrics in this notebook (F1, cross-validation, `scale_pos_weight`) are explicitly configured 
to treat **class 0 (phishing) as the positive class** — this matters for a phishing-detection 
task because missing a real phishing site (false negative on the phishing class) is more 
costly than a false alarm on a legitimate one. Leaving sklearn/XGBoost at their defaults 
would silently optimize for detecting *legitimate* sites instead.

In [2]:
from pathlib import Path
import pandas as pd

search_roots = [Path.cwd()]
if Path("/kaggle/input").exists():
    search_roots.append(Path("/kaggle/input"))

candidates = []

for root in search_roots:
    candidates.extend(root.glob("**/phishing.csv"))

if not candidates:
    raise FileNotFoundError(
        "phishing.csv not found! If on Kaggle, confirm the dataset is added via"
        "'+ Add Input' in the right sidebar, then re-run the /kaggle/input walk above."
    )

data_path = candidates[0]
print(f"Using dataset path: {data_path}")

#Load the CSV into a Pandas dataframe
df = pd.read_csv(data_path, usecols=["HTTPS", "AnchorURL", "PrefixSuffix-", "ServerFormHandler", "WebsiteTraffic", "GoogleIndex", "DNSRecording", "class"])
print(f'\nRaw shape: {df.shape}')

#Remap -1 to 0 and leave 1 as is for "class" feature
# 1 = legitimate, 0 = phishing
df["class"] = df["class"].map({1:1, -1:0})

# Subset X to selected features: HTTPS, AnchorURL, PrefixSuffix-, ServerFormHandler, WebsiteTraffic, GoogleIndex,DNSRecording
# Features were selected based on the results of the Feature Importance Extraction done in at: 
# https://www.kaggle.com/code/azimehobadiah/url-phising-dataset-feature-importance/
selected_features = ["HTTPS", "AnchorURL", "PrefixSuffix-", "ServerFormHandler", "WebsiteTraffic", "GoogleIndex", "DNSRecording"]
X = df[selected_features]
y = df["class"].astype(int)

# Print selected features
print("\n============== Selected Features =============")
for feature in selected_features:
    print(feature)

#Print Shape of selected features DataFrame
print("\nX Shape:", X.shape)

# Class Balance
print("\n============= Class balance: ==============")
class_counts = y.value_counts()
class_percentages = y.value_counts(normalize=True) * 100

for label in class_counts.index:
    count = class_counts[label]
    pct = class_percentages[label]
    print(f'\nClass {label}: {count} samples ({pct:.2f}%)')

class_counts

Using dataset path: /kaggle/input/datasets/eswarchandt/phishing-website-detector/phishing.csv

Raw shape: (11054, 8)

============== Selected Features =============
HTTPS
AnchorURL
PrefixSuffix-
ServerFormHandler
WebsiteTraffic
GoogleIndex
DNSRecording

X Shape: (11054, 7)

============= Class balance: ==============

Class 1: 6157 samples (55.70%)

Class 0: 4897 samples (44.30%)


class
1    6157
0    4897
Name: count, dtype: int64

## 3. Train/Test Split

An 80/20 stratified split is used to preserve the original class proportions (~55.7% 
legitimate, ~44.3% phishing) in both the training and test sets. `random_state=42` is 
fixed for reproducibility across this and all downstream notebooks in the project.

In [3]:
# Split Dataframe into training and testing sets
from sklearn.model_selection import train_test_split

# Stratified 80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("\nTraining set shape:", X_train.shape, y_train.shape)
print("\nTesting set shape:", X_test.shape, y_test.shape)

# Print class balance in split
for name, y_split in [("Train", y_train), ("Test", y_test)]:
    counts = y_split.value_counts().sort_index()
    percentages = y_split.value_counts(normalize=True).sort_index().mul(100).round(2)
    print(f'\n{name} class balance:')
    print(pd.concat([counts, percentages], axis=1, keys=['count', 'percentage']))


Training set shape: (8843, 7) (8843,)

Testing set shape: (2211, 7) (2211,)

Train class balance:
       count  percentage
class                   
0       3918       44.31
1       4925       55.69

Test class balance:
       count  percentage
class                   
0        979       44.28
1       1232       55.72


## 4. XGBoost — Hyperparameter Tuning via GridSearchCV

XGBoost is tuned over `n_estimators`, `max_depth`, and `learning_rate` using 5-fold 
`GridSearchCV`. Scoring uses **F1 with class 0 (phishing) as the positive class** — chosen 
over raw accuracy because it balances precision and recall for the class we actually care 
about detecting, rather than rewarding the model for simply predicting the majority class.

`scale_pos_weight` is computed from the training set's class ratio and passed directly into 
the base estimator to correct for the mild class imbalance (~56/44) — XGBoost does not apply 
this automatically.

The best estimator is then re-evaluated with 5-fold cross-validation to report a mean ± 
standard deviation F1 score, giving a sense of model stability rather than a single lucky split.

In [4]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import make_scorer, f1_score
import numpy as np

# Define the parameter grid for XGBClassifier
param_grid_xgb = {
    "n_estimators":[100, 300, 500],
    "max_depth": [3, 6, 9],
    "learning_rate": [0.01, 0.1, 0.2],
}

# Compute scale_pos_weight (majority count / minority count) so XGBoost corrects for
# the mild class imbalance in the training set
majority_count = y_train.value_counts()[1]
minority_count = y_train.value_counts()[0]
scale_pos_weight = majority_count / minority_count

# Score F1 with class 0 (phishing) as the positive class, since that's the class we
# care most about detecting correctly
f1_phishing_scorer = make_scorer(f1_score, pos_label=0)

# Initialize the base estimator, passing scale_pos_weight to correct for imbalance
xgb_base = XGBClassifier(
    random_state=42,
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight
)

# initialize Grid Search
xgb_grid_search = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid_xgb,
    cv=5,
    scoring=f1_phishing_scorer,
    n_jobs=-1,
    verbose=1
)

# fit training data to grid search

xgb_grid_search.fit(X_train, y_train)

# Print best parameters and best score
print(f"\nBest Training parameters: {xgb_grid_search.best_params_}")
print(f"\nBest f1 score (phishing class): {xgb_grid_search.best_score_}")

# Save best estimator model to a variable
best_estimator = xgb_grid_search.best_estimator_

# Cross-evalute the best estimator
cv_scores = cross_val_score(
    estimator=best_estimator,
    X=X_train,
    y=y_train,
    cv=5,
    scoring=f1_phishing_scorer,
    n_jobs=-1
)

# Print the fold-by-fold and summary results
print("\n================== Best Estimator CV Fold Scores (phishing-class F1) =======================")
for i, score in enumerate(cv_scores):
    print(f"\nFold {i+1}:{score:.4f}")

print("\n================== Summary Performance =========================")
print(f'\nMean Score: {np.mean(cv_scores):.4f}')
print(f"Standard Deviation: {np.std(cv_scores):.4f}")
print(f"Final Report: {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f} ")

xgb_model = best_estimator

Fitting 5 folds for each of 27 candidates, totalling 135 fits

Best Training parameters: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 300}

Best f1 score (phishing class): 0.9175048627399696

================== Best Estimator CV Fold Scores (phishing-class F1) =======================

Fold 1:0.9173

Fold 2:0.9126

Fold 3:0.9263

Fold 4:0.9136

Fold 5:0.9177

================== Summary Performance =========================

Mean Score: 0.9175
Standard Deviation: 0.0048
Final Report: 0.9175 ± 0.0048 


## 5. Random Forest — Hyperparameter Tuning via GridSearchCV

Random Forest is tuned over `n_estimators`, `max_depth`, and `min_samples_split` using the 
same 5-fold `GridSearchCV` setup as XGBoost, scored on **F1 with class 0 (phishing) as the 
positive class**, for a like-for-like comparison between the two algorithms. 
`class_weight="balanced"` is set to account for the mild class imbalance in the dataset. 
As with XGBoost, the best estimator is cross-validated to report mean ± standard deviation F1.

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

# Define parameter grid for the RF Classifier
param_grid_rf = {
    "n_estimators": [100, 300, 500],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5, 10]
}

# Initialize the RF base estimator with class weight=balanced to address mild class imbalance
rf_base = RandomForestClassifier(
    class_weight="balanced",
    random_state=42
)

# Initialise the Grid Search (uses the same f1_phishing_scorer defined in the XGBoost cell,
# so both models are compared on identical scoring criteria)
rf_grid_search = GridSearchCV(
    estimator=rf_base,
    cv=5,
    scoring=f1_phishing_scorer,
    n_jobs=-1,
    param_grid=param_grid_rf
)

# Fit the model with training data
rf_grid_search.fit(X_train, y_train)

# Print best parameters and best score
print(f"\nBest Training parameters: {rf_grid_search.best_params_}")
print(f"\nBest f1 score (phishing class): {rf_grid_search.best_score_}")

# Save best estimator model to a variable
best_estimator_rf = rf_grid_search.best_estimator_


# Cross-evalute the best estimator
cv_scores_rf = cross_val_score(
    estimator=best_estimator_rf,
    X=X_train,
    y=y_train,
    cv=5,
    scoring=f1_phishing_scorer,
    n_jobs=-1
)

# Print the fold-by-fold and summary results
print("\n================== Best Estimator CV Fold Scores (phishing-class F1) =======================")
for i, score in enumerate(cv_scores_rf):
    print(f"\nFold {i+1}:{score:.4f}")

print("\n================== Summary Performance =========================")
print(f'\nMean Score: {np.mean(cv_scores_rf):.4f}')
print(f"Standard Deviation: {np.std(cv_scores_rf):.4f}")
print(f"Final Report: {np.mean(cv_scores_rf):.4f} ± {np.std(cv_scores_rf):.4f} ")

rf_model = best_estimator_rf


Best Training parameters: {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 100}

Best f1 score (phishing class): 0.9185351847296752

================== Best Estimator CV Fold Scores (phishing-class F1) =======================

Fold 1:0.9232

Fold 2:0.9120

Fold 3:0.9248

Fold 4:0.9136

Fold 5:0.9191

================== Summary Performance =========================

Mean Score: 0.9185
Standard Deviation: 0.0051
Final Report: 0.9185 ± 0.0051 


## 6. Training-Set Sanity Check

This is **not** the formal model evaluation — it's a quick check on the training set to 
confirm neither model is badly underfitting before moving to held-out test set evaluation. 
Formal evaluation (test-set metrics, confusion matrices, ROC-AUC) is handled in a separate 
evaluation notebook to keep this stage focused purely on training.

F1 here is also computed with class 0 (phishing) as the positive class, consistent with the 
scoring convention used during tuning above.

In [6]:
from sklearn.metrics import accuracy_score, f1_score

rf_train_pred = rf_model.predict(X_train)
rf_accuracy = accuracy_score(y_train, rf_train_pred)
rf_f1 = f1_score(y_train, rf_train_pred, pos_label=0)

xgb_train_pred = xgb_model.predict(X_train)
xgb_accuracy = accuracy_score(y_train, xgb_train_pred)
xgb_f1 = f1_score(y_train, xgb_train_pred, pos_label=0)

# Print Accuracy and F1 scores of RF model
print(f'\nRF Accuracy Score: {rf_accuracy*100:.2f}%')
print(f"RF F1 Score (phishing class): {rf_f1*100:.2f}%")

# Print Accuracy and F1 scores of XGB model
print(f'\nXGB Accuracy Score: {xgb_accuracy*100:.2f}%')
print(f"XGB F1 Score (phishing class): {xgb_f1*100:.2f}%")


RF Accuracy Score: 92.93%
RF F1 Score (phishing class): 92.07%

XGB Accuracy Score: 92.84%
XGB F1 Score (phishing class): 91.89%


## 7. Export Trained Models and Evaluation Artifacts

Both tuned models are serialized with `joblib` for reuse in the evaluation notebook, so 
evaluation runs against the exact same trained models rather than retraining from scratch. 
The held-out test set (`X_test`, `y_test`) is also saved so the evaluation notebook uses the 
identical split produced here rather than re-splitting the data, and the best hyperparameters 
for both models are saved as JSON for the methodology write-up.

In [7]:
import joblib
import json

# Export trained models with joblib
joblib.dump(xgb_model, '/kaggle/working/xgb_model.joblib')
joblib.dump(rf_model, '/kaggle/working/rf_model.joblib')

# Export the exact held-out test set used here, so the evaluation notebook doesn't re-split
X_test.to_csv('/kaggle/working/X_test.csv', index=False)
y_test.to_csv('/kaggle/working/y_test.csv', index=False)

# Export best hyperparameters for both models, for the methodology write-up
best_params = {
    "xgb_best_params": xgb_grid_search.best_params_,
    "rf_best_params": rf_grid_search.best_params_
}
with open('/kaggle/working/best_params.json', 'w') as f:
    json.dump(best_params, f, indent=2)

print("Saved: xgb_model.joblib, rf_model.joblib, X_test.csv, y_test.csv, best_params.json")

Saved: xgb_model.joblib, rf_model.joblib, X_test.csv, y_test.csv, best_params.json
